In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [10]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient
import os
load_dotenv()
openai_client = OpenAI()

In [11]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results = 5,
        boost_dict= {"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict = {"course": "llm-zoomcamp"}
    )


In [12]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt= instructions,
    llm_client=OpenAIClient(model ="openai/gpt-oss-120b")
)


In [13]:
rec = ground_truth[0]

result = runner.loop(prompt = rec['question'])

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [14]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it okay to join the course late if I just found it now?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kxpawt8mf59sxtd7wenep7x0', summary=[], type='reasoning', content=[Content(text='We need to answer based on FAQ search. Use search tool to find relevant info: "join the course late", "late enrollment", "starting late", etc.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join the course late"}', call_id='fc_65fc0706-3170-419c-944e-ca95c24fb975', name='search', type='function_call', id='fc_65fc0706-3170-419c-944e-ca95c24fb975', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_65fc0706-3170-419c-944e-ca95c24fb97

In [18]:
def extract_tool_calls(messages):
    tool_calls = []
    for message in messages:
        if isinstance(message, dict):
            continue
        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })
    return tool_calls

In [19]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search', 'arguments': '{"query":"join the course late"}'}]

In [20]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [21]:
agent_result = {
    "question": rec['question'],
    "asnwer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

AttributeError: 'NoneType' object has no attribute 'total_cost'

In [22]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=1) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'o

KeyboardInterrupt: 

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


KeyboardInterrupt: 

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [ ]:
df_agent = pd.DataFrame(agent_answers)

In [ ]:
df_agent["cost"].sum()

NameError: name 'df_agent' is not defined

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'o

In [ ]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [26]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [ ]:
agent_answers

[{'question': 'Can I take this course at my own pace and still receive a certificate at the end?',
  'answer_agent': 'No — for this course, you can follow it in self-paced mode, but you **won’t receive a certificate** that way.\n\nTo get a certificate, you need to finish the course with the **live cohort**, since the certificate depends on participating in the capstone peer-review process, which is only available while the course is running.\n\nIf you want, I can also explain the certificate requirements or how the capstone/peer-review process works.',
  'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [29]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description = "Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description = "'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal['good', 'bad'] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [30]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [ ]:
import json
from evaluation_utils import calc_total_price , llm_structured_retry


def evaluate_agent_answer(rec, model = "openai/gpt-oss-120b" ):
    tool_calls = rec['tool_calls']

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model = model
    )

    return result , usage

/workspaces/LLM-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [34]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval


Rate limit or API error encountered. Retrying in 15 seconds... (Error: Error code: 400 - {'error': {'message': '`reasoning_effort` must be one of `low`, `medium`, or `high`', 'type': 'invalid_request_error'}})


KeyboardInterrupt: 

In [35]:

def judge_agent_record(rec):
    agent_eval , usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [ ]:
calc_total_price(usages)

In [ ]:
df_agent_eval["answer_score"].value_counts()

In [ ]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)